# Final Agent Evaluation

Evaluates all agents on the same predetermined surface views for fair comparison.

In [ ]:
import importlib
import gc
import random
import env.afm_env
importlib.reload(env.afm_env)
from env.afm_env import AfmEnvironment

from stable_baselines3 import SAC, PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from multiprocessing import Pool
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import pickle
import torch

## Configuration

Specify surfaces, agents, and evaluation parameters here.

In [ ]:
# ── Surfaces to evaluate on ──
SURFACES = [
    "environments/pt_111_2v_1a",
    "environments/pt_111_1a",
    "environments/pt_111_2v",
    "environments/pt_111_3v_1a",
    "environments/pt_111_plateau"
]

# ── Agents to evaluate ──
# Each entry: (display_name, model_folder_path)
# The algorithm (SAC/PPO) is inferred from the folder name.
# num_historic_data is inferred from hist400/hist40 in the folder name.
AGENTS = [
    # PPO eps=0.01
    ("PPO cnn eps0.01 #1", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.01_cnn_2026-04-10_19-39-26"),
    ("PPO cnn eps0.01 #2", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.01_cnn_2026-04-10_19-39-32"),
    ("PPO cnn eps0.01 #3", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.01_cnn_2026-04-10_19-39-36"),
    ("PPO cnn eps0.01 #4", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.01_cnn_2026-04-10_19-39-40"),
    ("PPO cnn eps0.01 #5", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.01_cnn_2026-04-10_19-39-44"),
    ("PPO cnn eps0.01 #6", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.01_cnn_2026-04-10_19-39-49"),
    # # PPO eps=0.05
    ("PPO cnn eps0.05 #1", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.05_cnn_2026-04-10_10-47-10"),
    ("PPO cnn eps0.05 #2", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.05_cnn_2026-04-10_10-47-14"),
    ("PPO cnn eps0.05 #3", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.05_cnn_2026-04-10_10-47-19"),
    ("PPO cnn eps0.05 #4", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.05_cnn_2026-04-10_10-47-30"),
    ("PPO cnn eps0.05 #5", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.05_cnn_2026-04-10_10-47-35"),
    ("PPO cnn eps0.05 #6", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.05_cnn_2026-04-10_10-47-39"),
    # PPO eps=0.1
    ("PPO cnn eps0.1 #1", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.1_cnn_2026-04-09_14-06-19"),
    ("PPO cnn eps0.1 #2", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.1_cnn_2026-04-09_14-06-20"),
    ("PPO cnn eps0.1 #3", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.1_cnn_2026-04-09_14-06-22"),
    ("PPO cnn eps0.1 #4", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.1_cnn_2026-04-09_14-07-02"),
    ("PPO cnn eps0.1 #5", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.1_cnn_2026-04-09_14-07-03"),
    ("PPO cnn eps0.1 #6", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.1_cnn_2026-04-09_14-07-04"),
    # PPO eps=0.2
    ("PPO cnn eps0.2 #1", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.2_cnn_2026-04-08_21-21-18"),
    ("PPO cnn eps0.2 #2", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.2_cnn_2026-04-08_21-21-19"),
    ("PPO cnn eps0.2 #3", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.2_cnn_2026-04-08_21-21-20"),
    ("PPO cnn eps0.2 #4", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.2_cnn_2026-04-08_21-21-22"),
    ("PPO cnn eps0.2 #5", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.2_cnn_2026-04-08_21-21-23"),
    ("PPO cnn eps0.2 #6", "evaluation/PPO/train_results/ppo_arch512x512_hist400_rc10.0_eps0.2_cnn_2026-04-08_21-21-25"),
    # SAC cnn
    ("SAC cnn #1", "evaluation/SAC/train_results/sac_arch512x512_hist400_rc10.0_cnn_2026-04-10_00-20-18"),
    ("SAC cnn #2", "evaluation/SAC/train_results/sac_arch512x512_hist400_rc10.0_cnn_2026-04-10_00-20-24"),
    ("SAC cnn #3", "evaluation/SAC/train_results/sac_arch512x512_hist400_rc10.0_cnn_2026-04-10_00-20-30"),
    ("SAC cnn #4", "evaluation/SAC/train_results/sac_arch512x512_hist400_rc10.0_cnn_2026-04-10_00-20-36"),
    ("SAC cnn #5", "evaluation/SAC/train_results/sac_arch512x512_hist400_rc10.0_cnn_2026-04-10_00-20-42"),
    ("SAC cnn #6", "evaluation/SAC/train_results/sac_arch512x512_hist400_rc10.0_cnn_2026-04-10_00-20-49"),
    # SAC (no cnn)
    ("SAC #1", "evaluation/SAC/train_results/sac_arch512x512_hist40_rc10.0_2026-04-09_12-27-08"),
    ("SAC #2", "evaluation/SAC/train_results/sac_arch512x512_hist40_rc10.0_2026-04-09_12-27-10"),
    ("SAC #3", "evaluation/SAC/train_results/sac_arch512x512_hist40_rc10.0_2026-04-09_12-27-14"),
    ("SAC #4", "evaluation/SAC/train_results/sac_arch512x512_hist40_rc10.0_2026-04-09_12-27-17"),
    ("SAC #5", "evaluation/SAC/train_results/sac_arch512x512_hist40_rc10.0_2026-04-09_12-27-20"),
    ("SAC #6", "evaluation/SAC/train_results/sac_arch512x512_hist40_rc10.0_2026-04-09_12-27-23"),
]

# ── Evaluation parameters ──
NUM_EVALUATIONS = 50  # episodes per agent
MAX_STEPS = 201 * 201 - 1
EVAL_BASE_SEED = 42  # global seed to make evaluation fully reproducible

# ── Env parameters (must match training) ──
ENV_KWARGS = dict(
    num_actions=1,
    include_image_in_info=False,
    base_reward=10,
    crash_reward=-100.0,
    sigma=2,
    height_offset_reward=0.4,
    reward_exponent=1,
)

# ── Views to plot detailed analysis for (indices into EVAL_VIEW_INDICES) ──
PLOT_VIEW_INDICES = [0, 10, 20, 30, 40]

## Pre-select evaluation views

Pick views evenly across all surfaces. These are fixed for all agents.

In [ ]:
EVAL_SEED = EVAL_BASE_SEED

# Count total views across all surfaces
surface_view_counts = []
for surface_dir in SURFACES:
    view_dirs = sorted([
        d for d in os.listdir(surface_dir)
        if d.startswith("view_") and os.path.isdir(os.path.join(surface_dir, d))
    ])
    surface_view_counts.append(len(view_dirs))
    print(f"{surface_dir}: {len(view_dirs)} views")

total_views = sum(surface_view_counts)
print(f"\nTotal views: {total_views}")

# Distribute evaluations evenly across surfaces, then pick random views within each
rng = np.random.default_rng(EVAL_SEED)
EVAL_VIEW_INDICES = []  # list of (surface_idx_in_SURFACES, view_idx_within_surface)
num_surfaces = len(SURFACES)
evals_per_surface = NUM_EVALUATIONS // num_surfaces
remainder = NUM_EVALUATIONS % num_surfaces

for surf_i, n_views in enumerate(surface_view_counts):
    n_evals = evals_per_surface + (1 if surf_i < remainder else 0)
    indices = rng.choice(n_views, size=n_evals, replace=False)
    for vi in sorted(indices):
        EVAL_VIEW_INDICES.append((surf_i, int(vi)))

print(f"\nSelected {len(EVAL_VIEW_INDICES)} evaluation view assignments (seed={EVAL_SEED})")
print(f"Views per surface: {[evals_per_surface + (1 if i < remainder else 0) for i in range(num_surfaces)]}")

## Helper functions

In [ ]:
def infer_agent_params(folder_name):
    """Infer algorithm class and num_historic_data from folder name."""
    basename = os.path.basename(folder_name)
    if basename.startswith("ppo"):
        algo_cls = PPO
    elif basename.startswith("sac"):
        algo_cls = SAC
    else:
        raise ValueError(f"Cannot infer algorithm from folder name: {basename}")

    if "hist400" in basename:
        num_historic_data = 400
    elif "hist40" in basename:
        num_historic_data = 40
    else:
        raise ValueError(f"Cannot infer num_historic_data from folder name: {basename}")

    return algo_cls, num_historic_data


def create_env_for_surface(surface_dir, num_historic_data, env_kwargs):
    env = AfmEnvironment(
        surface_configs=[{"data_dir_path": surface_dir}],
        num_historic_data=num_historic_data,
        **env_kwargs,
    )
    return env


def reset_vec_env_deterministic(raw_env, venv, view_idx, seed):
    """Reset a VecNormalize-wrapped env with fixed view + reproducible RNG seed."""
    raw_env.forced_view_idx = int(view_idx)
    try:
        venv.seed(int(seed))
        obs = venv.reset()
    finally:
        raw_env.forced_view_idx = None
    return obs


def _set_torch_determinism(seed):
    """Best-effort deterministic PyTorch setup for subprocess evaluation workers."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def _cleanup_surface_envs(surface_envs):
    """Close vec envs and raw envs to release resources in long parallel evals."""
    for raw_env, venv, _ in surface_envs.values():
        try:
            venv.close()
        except Exception:
            pass
        try:
            raw_env.close()
        except Exception:
            pass


def evaluate_agent(agent_name, agent_folder, eval_view_indices, surfaces, env_kwargs, max_steps, eval_base_seed):
    """Evaluate one agent over all assigned views. Designed to run in a subprocess."""
    algo_cls, num_historic_data = infer_agent_params(agent_folder)
    model_path = os.path.join(agent_folder, "best_model", "best_model.zip")
    vec_path   = os.path.join(agent_folder, "best_model", "best_vecnormalize.pkl")

    _set_torch_determinism(eval_base_seed)
    model = algo_cls.load(model_path)
    print(f"[{agent_name}] Starting (algo={algo_cls.__name__}, hist={num_historic_data}, seed={eval_base_seed})", flush=True)

    surface_envs = {}
    episodes = []

    try:
        for eval_i, (surf_i, view_idx) in enumerate(eval_view_indices):
            surface_dir = surfaces[surf_i]
            episode_seed = eval_base_seed + eval_i

            if surf_i not in surface_envs:
                raw_env = create_env_for_surface(surface_dir, num_historic_data, env_kwargs)
                venv = DummyVecEnv([lambda e=raw_env: e])
                venv = VecNormalize.load(vec_path, venv)
                venv.training = False
                venv.norm_reward = False
                surface_envs[surf_i] = (raw_env, venv, model)

            raw_env, venv, model = surface_envs[surf_i]
            obs = reset_vec_env_deterministic(raw_env, venv, view_idx=view_idx, seed=episode_seed)

            rewards_arr, action_arr, z_arr, dz_arr = [], [], [], []
            x_arr, y_arr, z_current_arr, z_opt_arr, zone_arr = [], [], [], [], []
            generated_image = None
            z_start = None

            for _ in range(max_steps):
                action, _ = model.predict(obs, deterministic=True)
                action_arr.append(float(np.asarray(action).reshape(-1)[0]))

                obs, rewards, dones, infos = venv.step(action)
                rewards_arr.append(float(rewards[0]))

                info0 = infos[0]
                if z_start is None:
                    z_start = float(info0.get("z", np.nan))
                z_arr.append(float(info0.get("z", np.nan)))
                dz_arr.append(float(obs["dz"][0][0]))
                x_arr.append(int(info0.get("x_pos", 0)))
                y_arr.append(int(info0.get("y_pos", 0)))
                z_current_arr.append(float(info0.get("z_current", np.nan)))
                z_opt_arr.append(float(info0.get("z_opt", np.nan)))
                zone_arr.append(info0.get("zone", "unknown"))

                if dones[0]:
                    break

            if hasattr(raw_env, "generated_image"):
                generated_image = np.array(raw_env.generated_image, copy=True)

            z_current_arr = np.array(z_current_arr, dtype=np.float32)
            z_opt_arr     = np.array(z_opt_arr, dtype=np.float32)
            deviation_arr = np.abs(z_current_arr - z_opt_arr)
            n_steps = len(zone_arr)
            zone_fractions = {
                "above_optimal": sum(z == "above_optimal" for z in zone_arr) / n_steps,
                "danger":        sum(z == "danger"        for z in zone_arr) / n_steps,
                "crash":         sum(z == "crash"         for z in zone_arr) / n_steps,
            }

            episodes.append({
                "rewards":               np.array(rewards_arr, dtype=np.float32),
                "total_reward":          float(np.sum(rewards_arr)),
                "actions":               np.array(action_arr, dtype=np.float32),
                "z_history":             np.array(z_arr, dtype=np.float32),
                "dz_history":            np.array(dz_arr, dtype=np.float32),
                "x_trajectory":          np.array(x_arr, dtype=np.int32),
                "y_trajectory":          np.array(y_arr, dtype=np.int32),
                "z_current_history":     z_current_arr,
                "z_opt_history":         z_opt_arr,
                "deviation_from_optimal": deviation_arr,
                "mean_deviation":        float(np.mean(deviation_arr)),
                "std_deviation":         float(np.std(deviation_arr)),
                "zone_fractions":        zone_fractions,
                "generated_image":       generated_image,
                "episode_length":        n_steps,
                "terminated_early":      n_steps < max_steps,
                "z_start":               z_start,
                "surface_idx":           surf_i,
                "surface_name":          surface_dir,
                "view_idx":              view_idx,
                "episode_seed":          episode_seed,
            })

            print(f"[{agent_name}] ep {eval_i+1}/{len(eval_view_indices)} | "
                  f"surface={surface_dir} view={view_idx} | "
                  f"seed={episode_seed} | "
                  f"reward={episodes[-1]['total_reward']:.1f} | "
                  f"steps={n_steps} | mean_dev={episodes[-1]['mean_deviation']:.4f}", flush=True)

        # Normalization statistics from the loaded VecNormalize
        any_venv = next(iter(surface_envs.values()))[1]
        norm_stats = {
            key: {"mean": rms.mean.tolist(), "var": rms.var.tolist()}
            for key, rms in any_venv.obs_rms.items()
        }
        norm_stats["_clip_obs"]    = float(any_venv.clip_obs)
        norm_stats["_clip_reward"] = float(any_venv.clip_reward)
        if any_venv.ret_rms is not None:
            norm_stats["_ret_mean"] = float(any_venv.ret_rms.mean)
            norm_stats["_ret_var"]  = float(any_venv.ret_rms.var)

        print(f"[{agent_name}] Done.", flush=True)
        return agent_name, episodes, norm_stats
    finally:
        _cleanup_surface_envs(surface_envs)
        gc.collect()


def evaluate_run(episode, ma_height_len=100, ma_action_len=100, ma_reward_len=100, title_prefix=""):
    """Plot detailed analysis of a single episode."""
    actions = episode["actions"]
    rewards = episode["rewards"]
    z_history = episode["z_history"]
    dz_history = episode["dz_history"]
    generated_image = episode.get("generated_image")

    if generated_image is not None:
        fig, ax = plt.subplots(figsize=(12, 10))
        im = ax.imshow(generated_image.T)
        fig.colorbar(im, ax=ax)
        ax.set_title(f"{title_prefix}Captured image")
        plt.show()
        plt.close(fig)

    n_steps = len(rewards)
    x_trajectory = episode.get("x_trajectory")
    y_trajectory = episode.get("y_trajectory")
    height_values = episode.get("z_current_history")
    if height_values is None:
        height_values = z_history + dz_history
    height_values = np.asarray(height_values, dtype=np.float32).flatten()

    if generated_image is not None:
        image_shape = generated_image.shape
    elif x_trajectory is not None and y_trajectory is not None and len(x_trajectory) > 0:
        image_shape = (int(np.max(x_trajectory)) + 1, int(np.max(y_trajectory)) + 1)
    else:
        image_shape = (201, 201)

    height_img = np.full(image_shape, np.nan, dtype=np.float32)
    if "z_start" in episode:
        height_img[0, 0] = episode["z_start"]
    if x_trajectory is not None and y_trajectory is not None:
        n_fill = min(len(x_trajectory), len(y_trajectory), len(height_values))
        height_img[x_trajectory[:n_fill], y_trajectory[:n_fill]] = height_values[:n_fill]
    fig, ax = plt.subplots()
    im = ax.imshow(height_img)
    ax.set_title(f"{title_prefix}Height of tip")
    fig.colorbar(im, ax=ax)
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(20, 10))
    moving_avg = np.convolve(height_values.flatten(), np.ones(ma_height_len)/ma_height_len, mode='valid')
    ax.plot(moving_avg)
    ax.set_title(f"{title_prefix}Moving average ({ma_height_len}) of height")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(20, 10))
    moving_avg = np.convolve(np.array(actions).flatten(), np.ones(ma_action_len)/ma_action_len, mode='valid')
    ax.plot(moving_avg)
    ax.set_title(f"{title_prefix}Moving average ({ma_action_len}) of actions")
    plt.show()
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(20, 10))
    moving_avg = np.convolve(np.array(rewards).flatten(), np.ones(ma_reward_len)/ma_reward_len, mode='valid')
    ax.plot(moving_avg)
    ax.set_title(f"{title_prefix}Moving average ({ma_reward_len}) of rewards (AVG={np.mean(rewards):.2f}, STD={np.std(rewards):.2f}, TOTAL={np.sum(rewards):.2f})")
    plt.show()
    plt.close(fig)
    plt.close('all')

## Run evaluation for all agents

In [ ]:
N_WORKERS = 18 # min(len(AGENTS), os.cpu_count() or 1)
print(f"Running {len(AGENTS)} agents on {N_WORKERS} workers in parallel.")

args = [
    (name, folder, EVAL_VIEW_INDICES, SURFACES, ENV_KWARGS, MAX_STEPS, EVAL_BASE_SEED)
    for i, (name, folder) in enumerate(AGENTS)
]

# maxtasksperchild=1 forces process recycling, preventing slow memory buildup in long runs.
with Pool(processes=N_WORKERS, maxtasksperchild=1) as pool:
    raw_results = pool.starmap(evaluate_agent, args)

# Preserve AGENTS order (Pool.starmap preserves order, but be explicit)
all_results    = {}
norm_stats_all = {}
for agent_name, episodes, norm_stats in raw_results:
    all_results[agent_name]    = episodes
    norm_stats_all[agent_name] = norm_stats

# Release temporary container once unpacked
del raw_results
gc.collect()

print("\nAll agents evaluated.")

In [ ]:
SAVE_DIR = "evaluation/results"
os.makedirs(SAVE_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Full results + norm stats as a single pickle
pkl_path = os.path.join(SAVE_DIR, f"results_{timestamp}.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump({"all_results": all_results, "norm_stats_all": norm_stats_all}, f)
print(f"Saved full results to {pkl_path}")

# Per-episode summary CSV
rows = []
for agent_name, episodes in all_results.items():
    for ep in episodes:
        rows.append({
            "agent":            agent_name,
            "surface":          ep["surface_name"],
            "view_idx":         ep["view_idx"],
            "z_start":          ep["z_start"],
            "total_reward":     ep["total_reward"],
            "episode_length":   ep["episode_length"],
            "terminated_early": ep["terminated_early"],
            "mean_deviation":   ep["mean_deviation"],
            "std_deviation":    ep["std_deviation"],
            "pct_above_optimal": ep["zone_fractions"]["above_optimal"] * 100,
            "pct_danger":        ep["zone_fractions"]["danger"] * 100,
            "pct_crash":         ep["zone_fractions"]["crash"] * 100,
        })

csv_path = os.path.join(SAVE_DIR, f"summary_{timestamp}.csv")
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f"Saved summary CSV to {csv_path}")

# Norm stats as a separate CSV for easy inspection
norm_rows = []
for agent_name, stats in norm_stats_all.items():
    for key, val in stats.items():
        if key.startswith("_"):
            norm_rows.append({"agent": agent_name, "key": key, "value": val})
        else:
            for i, (m, v) in enumerate(zip(val["mean"], val["var"])):
                norm_rows.append({"agent": agent_name, "key": key, "index": i, "mean": m, "var": v})

norm_csv_path = os.path.join(SAVE_DIR, f"norm_stats_{timestamp}.csv")
pd.DataFrame(norm_rows).to_csv(norm_csv_path, index=False)
print(f"Saved norm stats CSV to {norm_csv_path}")

## Summary statistics

In [ ]:
print(f"{'Agent':<45} {'Avg Reward':>12} {'Std Reward':>12} {'Avg Steps':>10} {'Early Term':>10} {'Completion':>10} {'Mean Dev':>10} {'Std Dev':>10} {'%AboveOpt':>10} {'%Danger':>10}")
print("-" * 155)

for agent_name, episodes in all_results.items():
    total_rewards = [ep["total_reward"] for ep in episodes]
    lengths = [ep["episode_length"] for ep in episodes]
    early = sum(1 for ep in episodes if ep["terminated_early"])
    mean_devs = [ep["mean_deviation"] for ep in episodes]
    std_devs = [ep["std_deviation"] for ep in episodes]
    pct_above = np.mean([ep["zone_fractions"]["above_optimal"] for ep in episodes]) * 100
    pct_danger = np.mean([ep["zone_fractions"]["danger"] for ep in episodes]) * 100

    print(f"{agent_name:<45} "
          f"{np.mean(total_rewards):>12.1f} "
          f"{np.std(total_rewards):>12.1f} "
          f"{np.mean(lengths):>10.0f} "
          f"{early:>10d} "
          f"{(len(episodes) - early) / len(episodes) * 100:>9.1f}% "
          f"{np.mean(mean_devs):>10.4f} "
          f"{np.mean(std_devs):>10.4f} "
          f"{pct_above:>9.1f}% "
          f"{pct_danger:>9.1f}%")

# Box plot of total rewards per agent
fig, ax = plt.subplots(figsize=(max(10, 3 * len(all_results)), 6))
data = [[ep["total_reward"] for ep in eps] for eps in all_results.values()]
ax.boxplot(data, labels=list(all_results.keys()), vert=True, showfliers=False)
ax.set_ylabel("Total Reward")
ax.set_title("Total Reward Distribution per Agent")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Episode length distribution
fig, ax = plt.subplots(figsize=(max(10, 3 * len(all_results)), 6))
data = [[ep["episode_length"] for ep in eps] for eps in all_results.values()]
ax.boxplot(data, labels=list(all_results.keys()), vert=True, showfliers=False)
ax.set_ylabel("Episode Length")
ax.set_title("Episode Length Distribution per Agent")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Mean deviation from optimal height
fig, ax = plt.subplots(figsize=(max(10, 3 * len(all_results)), 6))
data = [[ep["mean_deviation"] for ep in eps] for eps in all_results.values()]
ax.boxplot(data, labels=list(all_results.keys()), vert=True, showfliers=False)
ax.set_ylabel("Mean |z - z_opt| (Å)")
ax.set_title("Mean Deviation from Optimal Height per Agent")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Zone fraction stacked bar
agent_names = list(all_results.keys())
pct_above_opt = [np.mean([ep["zone_fractions"]["above_optimal"] for ep in eps]) * 100 for eps in all_results.values()]
pct_danger    = [np.mean([ep["zone_fractions"]["danger"]        for ep in eps]) * 100 for eps in all_results.values()]
pct_crash     = [np.mean([ep["zone_fractions"]["crash"]         for ep in eps]) * 100 for eps in all_results.values()]

x = np.arange(len(agent_names))
fig, ax = plt.subplots(figsize=(max(10, 3 * len(all_results)), 6))
ax.bar(x, pct_above_opt, label="Above optimal", color="steelblue")
ax.bar(x, pct_danger,    bottom=pct_above_opt, label="Danger zone", color="orange")
ax.bar(x, pct_crash,     bottom=[a + d for a, d in zip(pct_above_opt, pct_danger)], label="Crash", color="red")
ax.set_xticks(x)
ax.set_xticklabels(agent_names, rotation=45, ha="right")
ax.set_ylabel("% of steps")
ax.set_title("Zone Distribution per Agent (mean over episodes)")
ax.legend()
plt.tight_layout()
plt.show()

## Best run per agent + predetermined view details

In [ ]:
for agent_name, episodes in all_results.items():
    print(f"\n{'='*60}")
    print(f"Agent: {agent_name}")
    print(f"{'='*60}")

    # Best run (highest total reward)
    best_idx = max(range(len(episodes)), key=lambda i: episodes[i]["total_reward"])
    best_ep = episodes[best_idx]
    print(f"\n--- Best run (episode {best_idx}, view {best_ep['view_idx']} on surface {SURFACES[best_ep['surface_idx']]}) ---")
    print(f"    Total reward: {best_ep['total_reward']:.1f}, Steps: {best_ep['episode_length']}, Early term: {best_ep['terminated_early']}")
    evaluate_run(best_ep, ma_height_len=1, ma_action_len=1, ma_reward_len=1,
                 title_prefix=f"[{agent_name} BEST] ")

    # Predetermined views
    for pi in PLOT_VIEW_INDICES:
        if pi >= len(episodes):
            continue
        ep = episodes[pi]
        surf_i, view_idx = ep["surface_idx"], ep["view_idx"]
        print(f"\n--- Eval #{pi} (surface={SURFACES[surf_i]}, view={view_idx}) ---")
        print(f"    Total reward: {ep['total_reward']:.1f}, Steps: {ep['episode_length']}, Early term: {ep['terminated_early']}")
        evaluate_run(ep, ma_height_len=1, ma_action_len=1, ma_reward_len=1,
                     title_prefix=f"[{agent_name} eval#{pi}] ")